# 1.03 — Conditionals — solutions

Lesson: [`lessons/1.03_conditionals.md`](../lessons/1.03_conditionals.md)

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — leap years

A year is a leap year if it's divisible by 4, **except** years divisible by 100, **unless** they're also divisible by 400. Write `is_leap_year(year)`.

In [ ]:
def is_leap_year(year):
    if year % 400 == 0:
        return True
    if year % 100 == 0:
        return False
    return year % 4 == 0

In [ ]:
check(is_leap_year, [((2024,), True), ((2026,), False), ((1900,), False), ((2000,), True)])

## Problem 2 — grades

`grade(mark)`: 80+ → `"A"`, 70–79 → `"B"`, 60–69 → `"C"`, 50–59 → `"D"`, below 50 → `"F"`. Marks can be decimals (e.g. 79.5 is a B).

In [ ]:
def grade(mark):
    if mark >= 80:
        return "A"
    if mark >= 70:
        return "B"
    if mark >= 60:
        return "C"
    if mark >= 50:
        return "D"
    return "F"

In [ ]:
check(grade, [((95,), "A"), ((80,), "A"), ((79.5,), "B"), ((60,), "C"), ((50,), "D"), ((12,), "F")])

## Problem 3 — triangles

`classify_triangle(a, b, c)` returns `"invalid"` if any side is ≤ 0 or the sides can't form a triangle (a side is ≥ the sum of the other two), otherwise `"equilateral"`, `"isosceles"` or `"scalene"`.

In [ ]:
def classify_triangle(a, b, c):
    if a <= 0 or b <= 0 or c <= 0:
        return "invalid"
    if a >= b + c or b >= a + c or c >= a + b:
        return "invalid"
    if a == b == c:
        return "equilateral"
    if a == b or b == c or a == c:
        return "isosceles"
    return "scalene"

In [ ]:
check(classify_triangle, [
    ((3, 3, 3), "equilateral"), ((3, 3, 5), "isosceles"), ((5, 3, 3), "isosceles"),
    ((3, 4, 5), "scalene"), ((1, 2, 3), "invalid"), ((0, 1, 1), "invalid"), ((2, 9, 3), "invalid"),
])

## Problem 4 — shipping fee

`shipping_fee(weight_kg, express)`:

- 500 LKR for anything up to and including 1 kg,
- plus 200 LKR for each **started** kg above 1 kg (1.2 kg → one extra kg; 3.0 kg → two extra kg),
- `express=True` doubles the total.

Return an `int`.

In [ ]:
import math

def shipping_fee(weight_kg, express):
    fee = 500
    if weight_kg > 1:
        fee += 200 * math.ceil(weight_kg - 1)
    if express:
        fee *= 2
    return fee

In [ ]:
check(shipping_fee, [
    ((0.5, False), 500), ((1, False), 500), ((1.2, False), 700), ((3.0, False), 900),
    ((3.01, False), 1100), ((2, True), 1400),
])

## Problem 5 ★ — progressive tax

Using these (made-up) monthly brackets, write `tax_payable(income)`:

| Slice of income | Rate |
|---|---|
| first 150,000 | 0% |
| next 83,333 | 6% |
| next 41,667 | 18% |
| everything above that | 24% |

Round to 2 decimals. Hint: work slice by slice; `min` and `max` are your friends.

In [ ]:
def tax_payable(income):
    # How much of the income falls inside each slice (never negative, never more than the slice size)
    slice_6 = min(max(income - 150_000, 0), 83_333)
    slice_18 = min(max(income - 233_333, 0), 41_667)
    slice_24 = max(income - 275_000, 0)
    return round(slice_6 * 0.06 + slice_18 * 0.18 + slice_24 * 0.24, 2)

In [ ]:
check(tax_payable, [
    ((100_000,), 0.0),
    ((150_000,), 0.0),
    ((200_000,), 3000.0),
    ((275_000,), 12500.04),
    ((500_000,), 66500.04),
])

## Problem 6 ★ — loan decision

`loan_decision(age, monthly_income, current_repayments, new_repayment)` returns `"decline"`, `"refer"` or `"approve"`:

1. `"decline"` if age < 18 or age > 70, or monthly income < 50,000.
2. Otherwise compute the repayment ratio = (current + new repayments) / income.
3. ratio ≤ 0.40 → `"approve"`; ratio ≤ 0.55 → `"refer"`; above that → `"decline"`.

In [ ]:
def loan_decision(age, monthly_income, current_repayments, new_repayment):
    if age < 18 or age > 70 or monthly_income < 50_000:
        return "decline"
    ratio = (current_repayments + new_repayment) / monthly_income
    if ratio <= 0.40:
        return "approve"
    if ratio <= 0.55:
        return "refer"
    return "decline"

In [ ]:
check(loan_decision, [
    ((30, 200_000, 20_000, 40_000), "approve"),
    ((30, 200_000, 50_000, 40_000), "refer"),
    ((30, 200_000, 80_000, 40_000), "decline"),
    ((17, 900_000, 0, 1_000), "decline"),
    ((45, 40_000, 0, 1_000), "decline"),
    ((70, 100_000, 0, 40_000), "approve"),
])